In [1]:
from osgeo import gdal
import os
import subprocess
import geopandas as gpd
from pathlib import Path
import pandas as pd
from tqdm import tqdm

In [13]:
def extract_houses(df, footprints, crs): 
    regions_gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkt(df['Boundary']), crs=crs) 

    # Transform regions to WGS 84 to match city boundary
    regions_gdf = regions_gdf.to_crs('EPSG:3857')

    # print(f'There are {len(footprints_in_image)} houses found')

    id = []
    centroid = []
    type = []
    for image in tqdm(regions_gdf.geometry):
        footprints_in_image = footprints[footprints.intersects(image)]
        for index, footprint in footprints_in_image.iterrows():
            footprint_in_image_bbox = footprint.geometry.intersection(image)
        
            # Calculate the areas
            original_area = footprint.geometry.area
            intersection_area = footprint_in_image_bbox.area
            
            # Calculate the percentage of the footprint area within the image bounding box
            percentage_in_image = (intersection_area / original_area) * 100
            
            # Only proceed if the percentage is greater than 70%
            if percentage_in_image <= 70:
                continue
            id.append(str(int(footprint['OBJECTID'])))
            type.append(footprint['F_TYPE'])
            centroid.append(footprint.geometry.centroid)
    return [id, type, centroid]

In [3]:
footprints = gpd.read_file('C:/Users/Lenovo/Desktop/python/ra/Solar-Panel/testing/PROPERTY_BOUNDARIES_WGS84.shp')
footprints = footprints[footprints['F_TYPE']== 'COMMON']

In [14]:
year = 2022
file = f'C:/Users/Lenovo/Desktop/python/ra/Solar-Panel/data/images_check/{year}_image_check.xlsx'
df = pd.read_excel(file)
infos = extract_houses(df, footprints, 'EPSG:2952')
houses = pd.DataFrame({
    'id': infos[0],
    'type': infos[1],
    'centroid': infos[2]
})
houses['year'] = year
# houses.to_excel(f'C:/Users/Lenovo/Desktop/python/ra/Solar-Panel/data/house_info/{year}_house_info.xlsx', index=False)

100%|██████████████████████████████████████████████████████████████████████████████| 2751/2751 [04:23<00:00, 10.46it/s]


In [9]:
houses[houses['type'] == 'COMMON']

,id,type,centroid,year
0,5159106,COMMON,POINT (-8864941.461872526 5425898.042156082),2022
1,5160524,COMMON,POINT (-8864855.751975631 5425906.452055884),2022
2,5150493,COMMON,POINT (-8865257.41540519 5426746.969971733),2022
3,5151397,COMMON,POINT (-8865069.852828464 5426695.443832925),2022
4,5151920,COMMON,POINT (-8864908.550045991 5426699.299902141),2022
...,...,...,...,...
562203,5047546,COMMON,POINT (-8808003.621983964 5433630.229666495),2022
562204,5032904,COMMON,POINT (-8808234.176795691 5434033.890816814),2022
562205,5047517,COMMON,POINT (-8807388.80979214 5433535.7737238575),2022
562206,5032904,COMMON,POINT (-8808234.176795691 5434033.890816814),2022


In [25]:
houses['id'].duplicated()

0         False
1         False
2         False
3         False
4         False
          ...  
470216    False
470217    False
470218    False
470219    False
470220    False
Name: id, Length: 470221, dtype: bool

Take 1 year, crop all the houses, then train 2 networks, one for houses, one for panels. Once we have all the houses, then we can apply this to filter other year's data